In [ ]:
import matplotlib.pyplot as plt

In [ ]:
# model = get_sc_diag0_cheetah_model()
import os
from lume_cheetah import LUMECheetahModel, CheetahSimulator
from virtual_accelerator.cheetah.transformer import SLACCheetahTransformer
from virtual_accelerator.cheetah.variables import get_variables_from_segment
from virtual_accelerator.utils.variables import (
    get_epics_to_name_mapping,
    split_control_and_observable,
)
from cheetah.accelerator import Segment
from cheetah.particles import ParticleBeam
import torch

incoming_beam = ParticleBeam.from_twiss(
    beta_x=torch.tensor(9.34),
    alpha_x=torch.tensor(-1.6946),
    emittance_x=torch.tensor(1e-7),
    beta_y=torch.tensor(9.34),
    alpha_y=torch.tensor(-1.6946),
    emittance_y=torch.tensor(1e-7),
    energy=torch.tensor(90e6),
)
incoming_beam.particle_charges = torch.tensor(1.0)

# Get path to lattice files
lcls_lattice = os.environ.get("LCLS_LATTICE")

# Create lattice from file
segment = Segment.from_lattice_json(os.path.join(lcls_lattice, "cheetah/sc_diag0.json"))


# Define the simulator using lattice and particle beam
simulator = CheetahSimulator(
    segment=segment,
    initial_beam_distribution=incoming_beam,
)

# get control system device to cheetah mapping
control_name_to_element_name = {
    k: v.lower() for k, v in get_epics_to_name_mapping().items()
}

# Create transformer that handles maps get/set calls
transformer = SLACCheetahTransformer(
    control_name_to_cheetah=control_name_to_element_name
)

# Get supported control system variables
# for the model
variables = get_variables_from_segment(segment)

# Define the controllable and observable variables
control_variables, observable_variables = split_control_and_observable(variables)

# Create model
model = LUMECheetahModel(
    simulator=simulator,
    transformer=transformer,
    control_variables=control_variables,
    observable_variables=observable_variables,
)

## Testing getting and setting control model variables 

In [ ]:
# Get variables objects
vars = list(model.supported_variables.keys())
model.supported_variables

In [ ]:
# Get variable values
model.get(vars)

In [ ]:
# get beam distribution on OTR screen
info = model.get(
    [
        "OTRS:DIAG0:420:Image:ArrayData",
    ]
)
fig, ax = plt.subplots()
ax.imshow(info["OTRS:DIAG0:420:Image:ArrayData"])

In [ ]:
# Reset model and verify variables when back to init values
model.reset()
model.get(vars)

## Test getting/setting composite lattice elements

In [ ]:
print(model.get(["QUAD:DIAG0:190:BCTRL"]))
getattr(simulator.segment, "qdg001")

In [ ]:
model.set({"QUAD:DIAG0:190:BCTRL": 6})
getattr(simulator.segment, "qdg001")

In [ ]:
model.get(["QUAD:DIAG0:190:BCTRL"])